In [9]:
import pyodbc
import pandas as pd
conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=10.142.16.246\ACCDWH;"
    r"DATABASE=AZTECA_STAGING;"
    r"Trusted_Connection=yes;"
)
query = """
SELECT
    s.SER AS [Nombre de Servicio],
    COALESCE(s.CUS, 'Ninguno') AS CUS
FROM DTM.SF_SERVICE_LEGV2 s
WHERE s.ESTADO_SER NOT IN (
    'Cancelado',
    'Error',
    'En Proceso',
    'Declinado'
)
AND s.FAMILIA_PRODUCTOS IN 
(
    'Internet',
    'Datos'
)
AND s.TIPO_PRODUCTO IN
(
    'L2',
    'L3'
)
ORDER BY
    s.SER;
"""
ActiveServices = pd.read_sql(query, conn)
ActiveServices.count()

E:\Users\jmartin\AppData\Local\Temp\ipykernel_26720\1472109134.py:33: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  ActiveServices = pd.read_sql(query, conn)


Nombre de Servicio    7953
CUS                   7953
dtype: int64

In [10]:
import os
import numpy as np
import pandas as pd

output_folder = "inputs"
os.makedirs(output_folder, exist_ok=True)
n_archivos = 6

total_filas = len(ActiveServices)
filas_por_archivo = total_filas // n_archivos
sobrantes = total_filas % n_archivos
inicio = 0
for i in range(1, n_archivos + 1):
    cantidad = filas_por_archivo + (1 if i <= sobrantes else 0)
    fin = inicio + cantidad
    chunk = ActiveServices.iloc[inicio:fin]
    archivo = os.path.join(
        output_folder,
        f"input{i}.xlsx"
    )
    chunk.to_excel(
        archivo,
        index=False
    )
    print(f"input{i}.xlsx → {len(chunk)} filas")
    inicio = fin
print("\nProceso terminado.")
print(f"Total filas: {total_filas}")
print(f"Archivos guardados en: {os.path.abspath(output_folder)}")

input1.xlsx → 1326 filas
input2.xlsx → 1326 filas
input3.xlsx → 1326 filas
input4.xlsx → 1325 filas
input5.xlsx → 1325 filas
input6.xlsx → 1325 filas

Proceso terminado.
Total filas: 7953
Archivos guardados en: e:\Users\jmartin\Desktop\PricingProjects\OrderingTrafficData\inputs


In [1]:
import pandas as pd
import numpy as np

file = "Historico Traficoo.xlsx"
excel_file = pd.ExcelFile(file)
print(excel_file.sheet_names)

['Noviembre', 'Diciembre', 'Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio', 'Agosto']


In [2]:
dfs = []
for sheet in excel_file.sheet_names:
    df = pd.read_excel(file, sheet_name=sheet)
    month = 0
    match sheet:
        case 'Noviembre':
            month = '01/11/2025'
        case 'Diciembre':
            month = '01/12/2025'
        case 'Enero':
            month = '01/01/2026'
        case 'Febrero':
            month = '01/02/2026'
        case 'Marzo':
            month = '01/03/2026'
        case 'Abril':
            month = '01/04/2026'
        case 'Mayo':
            month = '01/05/2026'
        case 'Junio':
            month = '01/06/2026'
        case 'Julio':
            month = '01/07/2026'
        case 'Agosto':
            month = '01/08/2026'
        case 'Septiembre':
            month = '01/09/2026'
    df['month'] = month
    dfs.append(df)

In [3]:
traffic = pd.concat(dfs, ignore_index=True)
columnas = [
    'SER',
    'Nombre del ítem',
    'Host',
    'IP Interface',
    'month',
    'value_avg bits received',
    'value_avg bits sent',
    'value_max bits received',
    'value_max bits sent'
]
traffic = traffic[columnas]
traffic.head(5)

,SER,Nombre del ítem,Host,IP Interface,month,value_avg bits received,value_avg bits sent,value_max bits received,value_max bits sent
0,SER-100962,Interface smartgroup7.1418110(smartgroup7.1418...,PE165CUCDOR,172.16.8.130,01/11/2025,102.88 Mbps,893.15 Mbps,219.87 Mbps,1859.51 Mbps
1,SER-101079,Interface Eth-Trunk2.1711(ECLCPPTRIDSER-101079),PE180FLODOR,172.16.8.156,01/11/2025,7.59 Mbps,41.99 Mbps,39.40 Mbps,51.10 Mbps
2,SER-101084,Interface Eth-Trunk1.1312134(ECLCPLERBASER-101...,PE180VENVST,172.16.8.43,01/11/2025,0.4805 Mbps,3.95 Mbps,22.98 Mbps,32.48 Mbps
3,SER-101490,Interface GigabitEthernet0/2/9.30305(ECLCPACAI...,PE180GMLVST,172.16.8.104,01/11/2025,0.0161 Mbps,0.1573 Mbps,0.9372 Mbps,8.77 Mbps
4,SER-101733,Interface smartgroup5.472134(smartgroup5.47213...,PE165TUNDOR,172.16.8.227,01/11/2025,1.91 Mbps,8.14 Mbps,38.93 Mbps,64.01 Mbps


In [4]:
columns = [
    'value_avg bits received',
    'value_avg bits sent',
    'value_max bits received',
    'value_max bits sent'
]

for column in columns:
    traffic[column] = pd.to_numeric(traffic[column].str.replace(' Mbps', '', regex=False),errors='coerce')
traffic.head(5)

,SER,Nombre del ítem,Host,IP Interface,month,value_avg bits received,value_avg bits sent,value_max bits received,value_max bits sent
0,SER-100962,Interface smartgroup7.1418110(smartgroup7.1418...,PE165CUCDOR,172.16.8.130,01/11/2025,102.8800,893.1500,219.8700,1859.51
1,SER-101079,Interface Eth-Trunk2.1711(ECLCPPTRIDSER-101079),PE180FLODOR,172.16.8.156,01/11/2025,7.5900,41.9900,39.4000,51.10
2,SER-101084,Interface Eth-Trunk1.1312134(ECLCPLERBASER-101...,PE180VENVST,172.16.8.43,01/11/2025,0.4805,3.9500,22.9800,32.48
3,SER-101490,Interface GigabitEthernet0/2/9.30305(ECLCPACAI...,PE180GMLVST,172.16.8.104,01/11/2025,0.0161,0.1573,0.9372,8.77
4,SER-101733,Interface smartgroup5.472134(smartgroup5.47213...,PE165TUNDOR,172.16.8.227,01/11/2025,1.9100,8.1400,38.9300,64.01


In [6]:
traffic.to_excel('trafico_limpio.xlsx', index=False)

In [5]:
import pyodbc
import pandas as pd
conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=10.142.16.246\ACCDWH;"
    r"DATABASE=AZTECA_STAGING;"
    r"Trusted_Connection=yes;"
)
query = """
SELECT 
    SER,
    CAPACIDADBPS
FROM [DTM].[SF_SERVICE_LEGV2]
"""
df_datamart = pd.read_sql(query, conn)
df_datamart = df_datamart.rename(columns={
    'SER': 'SER',
    'CAPACIDADBPS': 'CAPACIDAD'
})
print(df_datamart.head())

E:\Users\jmartin\AppData\Local\Temp\ipykernel_24756\239925001.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_datamart = pd.read_sql(query, conn)


              SER  CAPACIDAD
0  SER-0000000001       10.0
1  SER-0000000002       20.0
2  SER-0000000003       10.0
3  SER-0000000004       10.0
4  SER-0000000005        5.0


In [6]:
traffic_n = traffic.merge(
    df_datamart,
    on='SER',
    how='left'
)

In [7]:
for column in columns:
    new_name = column + '_n'
    traffic_n[new_name] = (
        traffic_n[column] / traffic_n['CAPACIDAD']
    )
traffic_n.head()

,SER,Nombre del ítem,Host,IP Interface,month,value_avg bits received,value_avg bits sent,value_max bits received,value_max bits sent,CAPACIDAD,value_avg bits received_n,value_avg bits sent_n,value_max bits received_n,value_max bits sent_n
0,SER-100962,Interface smartgroup7.1418110(smartgroup7.1418...,PE165CUCDOR,172.16.8.130,01/11/2025,102.8800,893.1500,219.8700,1859.51,4000.0,0.025720,0.223287,0.054968,0.464877
1,SER-101079,Interface Eth-Trunk2.1711(ECLCPPTRIDSER-101079),PE180FLODOR,172.16.8.156,01/11/2025,7.5900,41.9900,39.4000,51.10,50.0,0.151800,0.839800,0.788000,1.022000
2,SER-101084,Interface Eth-Trunk1.1312134(ECLCPLERBASER-101...,PE180VENVST,172.16.8.43,01/11/2025,0.4805,3.9500,22.9800,32.48,150.0,0.003203,0.026333,0.153200,0.216533
3,SER-101490,Interface GigabitEthernet0/2/9.30305(ECLCPACAI...,PE180GMLVST,172.16.8.104,01/11/2025,0.0161,0.1573,0.9372,8.77,60.0,0.000268,0.002622,0.015620,0.146167
4,SER-101733,Interface smartgroup5.472134(smartgroup5.47213...,PE165TUNDOR,172.16.8.227,01/11/2025,1.9100,8.1400,38.9300,64.01,450.0,0.004244,0.018089,0.086511,0.142244


In [8]:
traffic_n.to_excel('trafico_limpio_n.xlsx', index=False)